In [11]:
import pandas as pd
from scipy.stats import hypergeom
!pip install --upgrade scipy
from statsmodels.stats.multitest import multipletests

# the input gene list based on result of e2v in exp1-DB06637
input_genes = ['KCNF1', 'KCNG1', 'KCNG2', 'KCNG3', 'KCNG4', 'KCNS1', 'KCNS2', 'KCNS3',
               'KCNV1', 'KCNV2', 'KCNA1', 'KCNA10', 'KCNA2', 'KCNA3', 'KCNA4', 'KCNA5',
               'KCNA6', 'KCNA7', 'KCNB1', 'KCNB2', 'KCNC1', 'KCNC2', 'KCNC3', 'KCNC4',
               'KCND2', 'KCND3', 'KCNH1', 'KCNH2', 'KCNH3', 'KCNH4', 'KCNH5', 'KCNH6',
               'KCNH7', 'KCNH8', 'KCNQ1', 'KCNQ2', 'KCNQ3', 'KCNQ4', 'KCNQ5', 'SLC22A2']

# Load mapping of genes to body parts
background_body_part_mapping_path = '/Users/aorlenko/Downloads/3612referenceGenes_BP.csv'
#background_body_part_mapping = pd.read_csv(background_body_part_mapping_path)
background_body_part_mapping = pd.read_csv(background_body_part_mapping_path, header=0)
print(len(background_body_part_mapping.Gene.unique()))

missing_genes = [g for g in input_genes if g not in background_body_part_mapping['Gene'].values]
if missing_genes:
    print(f"Warning: {missing_genes} not in background set")
    input_genes = [g for g in input_genes if g not in missing_genes]
    print(f"Updated input_genes (after removing missing): {len(input_genes)} genes")
else:
    print("All input genes found in background set")

    
    # Count occurrences in input genes
input_path = '/Users/aorlenko/Downloads/DB06637_BP_Valuecounts.csv'
input_counts = pd.read_csv(input_path)
print(input_counts)

# Count occurrences in the entire background
background_path = '/Users/aorlenko/Downloads/3612referenceGenes_BP_Valuecounts.csv'
background_counts = pd.read_csv(background_path)
#print(background_counts)


### enrichment analysis test (hypergeometric)
# Total number of genes in the background
#total_genes = 3612
total_genes=len(background_body_part_mapping.Gene.unique())
print('background gene count ', total_genes)

# total genes in input_counts
x = len(input_genes)
print('updated gene set count', x)

# Define the body part of interest
body_parts = input_counts['BodyPart'].unique()
# Dictionary to store p-values for each body part
p_values = {}

# Perform hypergeometric test for each body part  this is an empty dictionary that can be used later
for body_part in body_parts:
    # Check input_counts for the current body part
    k = input_counts.loc[input_counts['BodyPart'] == body_part, 'Count'].values[0]
    print(f"k (input_counts for {body_part}): {k}")

    # Check background_counts for the current body part
    m = background_counts.loc[background_counts['BodyPart'] == body_part, 'Count'].values[0]
    print(f"m (background_counts for {body_part}): {m}")

    # Perform the hypergeometric test
    p_value = hypergeom.sf(k - 1, total_genes, m, x)
    #print(f"P-value for {body_part}: {p_value}")

    #Store the p-value in the dictionary i already created above  p_values = {}
    p_values[body_part] = p_value
    print(p_value)
    # Print the updated dictionary after each insertion
    #print(f"Updated p_values dictionary after adding {body_part}: {p_values}")

# Apply FDR correction
body_parts_list = list(p_values.keys())
p_values_list = list(p_values.values())

# Perform FDR correction (Benjamini-Hochberg method), in this code i only need corrected_p_values, and i use _ a placeholder to ignore other outputs(alphacBonf, reject..)
_, corrected_p_values, _, _ = multipletests(p_values_list, method='fdr_bh')

# Output corrected p-values
for i, body_part in enumerate(body_parts_list):
    print(f"Body part: {body_part}, Original P-value: {p_values[body_part]}, Corrected P-value: {corrected_p_values[i]}")


###save the output in a csv file
# Specify the output file
output_file = 'corrected_p_values.csv'

# Create a DataFrame
data = {
    'Body Part': body_parts_list,
    'Original P-value': [p_values[body_part] for body_part in body_parts_list],
    'Corrected P-value': corrected_p_values
}
df = pd.DataFrame(data)

# Save the DataFrame to CSV
df.to_csv(output_file, index=False)



3217
Updated input_genes (after removing missing): 34 genes
                  BodyPart  Count
0                    brain     27
1                    heart     21
2            adrenal gland     18
3                    blood     13
4                    liver     13
5                 midbrain     13
6        medulla oblongata     12
7               cerebellum     11
8                   testis     11
9      trigeminal ganglion     11
10                 urethra     10
11         pituitary gland     10
12                    lung      9
13           thyroid gland      8
14           telencephalon      8
15             spinal cord      7
16                  nipple      7
17       cardiac ventricle      7
18          cardiac atrium      7
19          adrenal cortex      6
20              myometrium      6
21        cortex of kidney      6
22         mammalian vulva      6
23             endometrium      6
24          adipose tissue      5
25                   ovary      5
26  saliva-secreting g

In [1]:
import pandas as pd
import ast  # Safer alternative to eval for parsing strings into Python objects
import os

# List of CSV files
csv_files = ['/Users/lib/Downloads/Drug_analysis/exp1/sub_median.csv']  # Replace with the names or paths of your CSV files

# Dictionary to store results for each CSV file
results = {}

# Loop through each CSV file
for csv_file in csv_files:
    if os.path.exists(csv_file):  # Check if the file exists
        # Read the CSV file
        df = pd.read_csv(csv_file)

        # Get unique values from the Name column
        unique_names = df['Name'].unique()

        # Dictionary to store genes for each Name in the current file
        genes_dict = {}
        for name in unique_names:
            filtered_df = df[df['Name'] == name]
            if not filtered_df.empty:
                # Parse the 'genes' column safely using ast.literal_eval
                genes_list = ast.literal_eval(filtered_df.iloc[0]['genes'])
                genes_dict[name] = genes_list

        # Store the results for the current file
        results[csv_file] = genes_dict
    else:
        print(f"File {csv_file} does not exist.")

# Display the results
for csv_file, genes_dict in results.items():
    print(f"Results for {csv_file}:")
    for name, genes in genes_dict.items():
        print(f"  Name: {name}, Genes: {genes}")

Results for /Users/lib/Downloads/Drug_analysis/exp1/sub_median.csv:
  Name: DB00363, Genes: ['HTR1A', 'HTR1B', 'HTR1D', 'HTR1E', 'HTR1F', 'HTR2A', 'HTR2B', 'HTR3A', 'HTR5A', 'HTR6', 'HTR7', 'ABCB1', 'UGT1A4', 'ADORA3', 'ADRA1A', 'ADRA1B', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRA2C', 'ADRB1', 'ADRB2', 'ALB', 'AOX1', 'CALY', 'CALM1', 'CHRM1', 'CHRM2', 'CHRM3', 'CHRM4', 'CHRM5', 'CYP1A1', 'CYP1A2', 'CYP2A6', 'CYP2C19', 'CYP2C8', 'CYP2C9', 'CYP2D6', 'CYP3A4', 'DRD1', 'DRD2', 'DRD3', 'DRD4', 'DRD5', 'FMO3', 'GABRA1', 'GABRB1', 'GABRG1', 'GABBR1', 'GABBR2', 'GSTP1', 'HRH1', 'HRH2', 'HRH3', 'HRH4', 'KCNH2', 'SIGMAR1', 'SLC22A1', 'SLC22A2', 'SLC22A3', 'SLC6A2', 'SLC6A4']
  Name: DB00257, Genes: ['HTR1A', 'HTR2A', 'HTR2B', 'HTR6', 'ABCB1', 'ABCB11', 'CCR4', 'CXCR1', 'DDIT4', 'FYN', 'ACHE', 'ADORA1', 'ADORA2A', 'ADORA3', 'ADRA1D', 'ADRA2A', 'ADRA2B', 'ADRB1', 'ADRB3', 'CACNA1C', 'CA2', 'CHRM1', 'CHRM2', 'CHRM3', 'CHRM4', 'CYP17A1', 'CYP19A1', 'CYP2A6', 'CYP2B6', 'CYP2C19', 'CYP2C8', 'CYP2C9', 'CYP2D6

In [2]:
genes_dict

{'DB00363': ['HTR1A',
  'HTR1B',
  'HTR1D',
  'HTR1E',
  'HTR1F',
  'HTR2A',
  'HTR2B',
  'HTR3A',
  'HTR5A',
  'HTR6',
  'HTR7',
  'ABCB1',
  'UGT1A4',
  'ADORA3',
  'ADRA1A',
  'ADRA1B',
  'ADRA1D',
  'ADRA2A',
  'ADRA2B',
  'ADRA2C',
  'ADRB1',
  'ADRB2',
  'ALB',
  'AOX1',
  'CALY',
  'CALM1',
  'CHRM1',
  'CHRM2',
  'CHRM3',
  'CHRM4',
  'CHRM5',
  'CYP1A1',
  'CYP1A2',
  'CYP2A6',
  'CYP2C19',
  'CYP2C8',
  'CYP2C9',
  'CYP2D6',
  'CYP3A4',
  'DRD1',
  'DRD2',
  'DRD3',
  'DRD4',
  'DRD5',
  'FMO3',
  'GABRA1',
  'GABRB1',
  'GABRG1',
  'GABBR1',
  'GABBR2',
  'GSTP1',
  'HRH1',
  'HRH2',
  'HRH3',
  'HRH4',
  'KCNH2',
  'SIGMAR1',
  'SLC22A1',
  'SLC22A2',
  'SLC22A3',
  'SLC6A2',
  'SLC6A4'],
 'DB00257': ['HTR1A',
  'HTR2A',
  'HTR2B',
  'HTR6',
  'ABCB1',
  'ABCB11',
  'CCR4',
  'CXCR1',
  'DDIT4',
  'FYN',
  'ACHE',
  'ADORA1',
  'ADORA2A',
  'ADORA3',
  'ADRA1D',
  'ADRA2A',
  'ADRA2B',
  'ADRB1',
  'ADRB3',
  'CACNA1C',
  'CA2',
  'CHRM1',
  'CHRM2',
  'CHRM3',
  'CHRM4',
 